In [2]:
import numpy as np 
from PIL import Image
import cv2
from ultralytics import YOLO

In [ ]:
model = YOLO('yolov8n.pt')
model("assets/human3.png", imgsz=(32, 32))[0].show() 



image 1/1 /Users/lalayants/Documents/itmo/ThesisCode/generate_video/assets/human3.png: 32x32 1 person, 3.1ms
Speed: 0.3ms preprocess, 3.1ms inference, 0.4ms postprocess per image at shape (1, 3, 32, 32)


In [23]:
import os
import cv2
import numpy as np
from PIL import Image
import math

# ---------------------------
# Parameters (customize as needed)
# ---------------------------
assets_folder = "assets"                   # Folder with your asset images
video_name = 'output_640_175.mp4'
video_width, video_height = 640, 640         # Video resolution
video_resolution = (video_width, video_height)
amount_of_objects = 5                        # Total number of objects to appear
duration_between_object_appearance = 1.0     # Seconds delay between object appearances
object_size = (175, 175)                     # Size (width, height) to which each asset will be resized
fps = 30                                     # Frames per second
period = 2.0                                 # Seconds for one full orbit (rotation period)

# Total video duration: after the last object appears.
total_duration = duration_between_object_appearance * amount_of_objects
total_frames = int(total_duration * fps)

# ---------------------------
# Load and preprocess all asset images
# ---------------------------
asset_files = [f for f in os.listdir(assets_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
if not asset_files:
    raise Exception("No asset images found in the assets folder.")

assets = []
for filename in asset_files:
    path = os.path.join(assets_folder, filename)
    try:
        pil_img = Image.open(path).convert("RGB")
        pil_img = pil_img.resize(object_size)
        # Convert from PIL (RGB) to OpenCV (BGR)
        cv_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        assets.append(cv_img)
    except Exception as e:
        print(f"Error loading {filename}: {e}")

if not assets:
    raise Exception("No valid asset images were loaded.")

# ---------------------------
# Orbit parameters: one big circle around the center of the video
# ---------------------------
center_x, center_y = video_width // 2, video_height // 2
margin = 10  # safety margin so objects don't clip the frame
# Calculate radius such that objects remain fully within the frame.
radius = min(video_width, video_height) / 2 - max(object_size) / 2 - margin

# ---------------------------
# Setup VideoWriter (output video: output.mp4)
# ---------------------------
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(video_name, fourcc, fps, video_resolution)

# ---------------------------
# Generate video frame-by-frame
# ---------------------------
for frame in range(total_frames):
    t = frame / fps  # Current time in seconds
    # Create a black background frame
    frame_img = np.zeros((video_height, video_width, 3), dtype=np.uint8)
    
    # Global rotation angle for the orbit (all objects share the same rotation speed)
    global_angle = 2 * math.pi * t / period
    
    for i in range(amount_of_objects):
        # Only display object i if its appearance time has passed
        if t < i * duration_between_object_appearance:
            continue
        
        # Each object is assigned a fixed angular offset so that they remain equally spaced.
        # Even if not all objects are active, their relative offsets are maintained.
        angle = global_angle + i * (2 * math.pi / amount_of_objects)
        
        # Compute the (x, y) position along the orbit
        x = int(center_x + radius * math.cos(angle))
        y = int(center_y + radius * math.sin(angle))
        # Adjust so that the image is centered at (x, y)
        x_top_left = x - object_size[0] // 2
        y_top_left = y - object_size[1] // 2
        
        # Safeguard to keep image within frame boundaries
        x_top_left = max(0, min(x_top_left, video_width - object_size[0]))
        y_top_left = max(0, min(y_top_left, video_height - object_size[1]))
        
        # Cycle through asset images if there are fewer than amount_of_objects
        asset_img = assets[i % len(assets)]
        
        # Place the asset image onto the frame
        frame_img[y_top_left:y_top_left + object_size[1], x_top_left:x_top_left + object_size[0]] = asset_img
    
    # Write the current frame to the video
    out.write(frame_img)

# Release the VideoWriter and finalize the video
out.release()
print("Video created: output.mp4")


Video created: output.mp4
